# Introduction to the OpenAI Responses API

The **Responses API** is OpenAI's primary API surface. It combines the strengths of Chat Completions and the old Assistants API, and it is where the modern features live: durable **conversations**, server-side **compaction**, response **phases**, hosted tools, connectors, and WebSocket streaming.

This notebook rebuilds the introduction around the **Conversations API** as the primary multi-turn pattern, and reframes Chat Completions as legacy.

> **Model update (GPT-5.5 refresh).** All `model=` calls target **`gpt-5.5`**. `reasoning_effort` is pinned explicitly where cost/latency matters (GPT-5.5 defaults to `medium`).
>
> **TODO — `gpt-5.5-instant` [unverified].** Light cells could use `gpt-5.5-instant`; its API name was unconfirmed at authoring time, so we use `gpt-5.5` throughout.

## Setup

In [ ]:
import os
import getpass

def _set_env(var: str):
    if not os.environ.get(var):
        os.environ[var] = getpass.getpass(f"{var}: ")

_set_env("OPENAI_API_KEY")

In [ ]:
!pip install --upgrade openai pandas jinja2 pydantic

## Initialize the Client

In [ ]:
from openai import OpenAI

client = OpenAI()
# Make sure your OPENAI_API_KEY environment variable is set

## Basic Text Response

A single, stateless call. `instructions` sets behavior; `input` is the user turn.

In [ ]:
response = client.responses.create(
    model="gpt-5.5",
    instructions="You are a coding assistant that talks like a pirate.",
    input="How do I check if a Python object is an instance of a class?",
    reasoning={"effort": "low"},
)

print(response.output_text)

## Multi-Turn with the Conversations API (the primary pattern)

Instead of manually re-sending history each turn, create a **durable conversation object** with `client.conversations.create()`. It stores messages, tool calls, and tool outputs under its own id. Pass that id on each `responses.create()` call and the model shares context automatically.

Conversations persist **indefinitely** (no 30-day TTL), unlike standalone `store: true` responses which expire after 30 days.

In [ ]:
# 1) Create a durable conversation
conversation = client.conversations.create()
print("conversation id:", conversation.id)

# 2) First turn
r1 = client.responses.create(
    model="gpt-5.5",
    conversation=conversation.id,
    input="I'm designing a URL shortener. What storage would you start with?",
    reasoning={"effort": "medium"},
)
print("Turn 1:", r1.output_text[:200], "...")

# 3) Second turn -- no manual history; the conversation id carries context
r2 = client.responses.create(
    model="gpt-5.5",
    conversation=conversation.id,
    input="Now how would I add custom vanity slugs to that design?",
    reasoning={"effort": "medium"},
)
print("\nTurn 2:", r2.output_text[:200], "...")

### Legacy chaining: `previous_response_id`

Before conversations, you chained turns with `previous_response_id`. It still works for `gpt-5.5`, but prefer conversations for durable, multi-turn state.

```python
follow_up = client.responses.create(
    model="gpt-5.5",
    input="...next turn...",
    previous_response_id=r1.id,   # legacy chaining
)
```

Billing note (both patterns): prior input tokens in the chain are re-billed as input on each turn.

## Server-Side Compaction (`/responses/compact`)

Long conversations grow expensive because prior tokens are re-billed each turn. The Responses API can **compact** context server-side -- shrinking what you send per turn while preserving meaning.

- Standalone endpoint: `client.responses.compact(...)`
- Inline: `context_management` / `compact_threshold` parameters that trigger compaction automatically once the context crosses a threshold.

> **TODO — parameter names [unverified].** The exact surface (`compact_threshold` vs `context_management`, and the `responses.compact` method path) may be renamed; confirm against the live changelog before publishing.

In [ ]:
# Standalone compaction of a long-running conversation's context.
# [unverified API surface] -- confirm method/params against the live changelog.

compacted = client.responses.compact(
    conversation=conversation.id,
)
print("Compacted context ready:", getattr(compacted, "id", compacted))

# Inline / automatic compaction: let the server compact once context grows large.
r3 = client.responses.create(
    model="gpt-5.5",
    conversation=conversation.id,
    input="Summarize the whole design so far in 5 bullets.",
    reasoning={"effort": "low"},
    # context_management={"compact_threshold": 200_000},  # [unverified] auto-compact knob
)
print(r3.output_text)

## Response Phases (`phase`: commentary vs final_answer)

The `phase` field labels assistant messages as either **`commentary`** (in-progress narration / thinking-out-loud) or **`final_answer`** (the deliverable). This lets a UI stream progress separately from the answer the user keeps.

In [ ]:
# Stream a response and separate commentary from the final answer by phase.
stream = client.responses.create(
    model="gpt-5.5",
    input="Plan and then write a Python function to validate an email address.",
    reasoning={"effort": "medium"},
    stream=True,
)

commentary, final = [], []
for event in stream:
    phase = getattr(event, "phase", None)          # [unverified] field name on events
    text = getattr(event, "delta", "") or ""
    if phase == "commentary":
        commentary.append(text)
    elif phase == "final_answer":
        final.append(text)

print("=== COMMENTARY (progress) ===")
print("".join(commentary)[:300])
print("\n=== FINAL ANSWER (deliverable) ===")
print("".join(final))

## Transport: WebSocket Mode (low-latency streaming)

For latency-sensitive apps, the Responses API supports a **WebSocket transport**. It keeps a connection-local cache so follow-up turns (via `previous_response_id`) continue with very low latency -- ideal for voice and live agent UIs.

We don't open a socket here (it needs a running event loop and the realtime/ws client), but the pattern is: open a WS connection, send response requests over it, and read streamed events back on the same connection. See the [OpenAI Realtime API docs](https://platform.openai.com/docs/guides/realtime) for a concrete WebSocket connection pattern.

## Image Analysis

In [ ]:
prompt = "Extract all the visual elements of this image into a bullet points list, just output that list."
img_url = "https://upload.wikimedia.org/wikipedia/commons/thumb/d/d5/2023_06_08_Raccoon1.jpg/1599px-2023_06_08_Raccoon1.jpg"

response = client.responses.create(
    model="gpt-5.5",
    input=[
        {
            "role": "user",
            "content": [
                {"type": "input_text", "text": prompt},
                {"type": "input_image", "image_url": f"{img_url}"},
            ],
        }
    ],
    reasoning={"effort": "low"},
)

print(response.output_text)

## Streaming Responses

In [ ]:
stream = client.responses.create(
    model="gpt-5.5",
    input="Write a one-sentence bedtime story about having pancakes as an afternoon snack.",
    reasoning={"effort": "none"},
    stream=True,
)

for event in stream:
    print(event)

## Structured Output with Pydantic

In [ ]:
from openai import OpenAI

def process_image(prompt, img_url, model="gpt-5.5"):
    response = client.responses.create(
        model=model,
        input=[
            {
                "role": "user",
                "content": [
                    {"type": "input_text", "text": prompt},
                    {"type": "input_image", "image_url": f"{img_url}"},
                ],
            }
        ],
        reasoning={"effort": "low"},
    )
    return response.output_text

In [ ]:
from pydantic import BaseModel, Field

class ImageElements(BaseModel):
    subject: str = Field(description="The main subject of the image")
    general_background_description: str = Field(description="A general description of the background of the image")


def extract_structured_text(output):
    responses = client.responses.parse(
        model="gpt-5.5",
        input=[
            {"role": "system", "content": "You extract the main subject and background contained in the input."},
            {"role": "user", "content": output},
        ],
        text_format=ImageElements,
    )
    return responses.output_parsed


photo_description = "A golden-brown pancake sits on a pristine white plate, its surface glistening with syrup and a pat of melting butter. The simple background highlights the pancake, making it the clear focus of the image."
extract_structured_text(photo_description)

In [ ]:
from IPython.display import Markdown

image_of_a_pancake = "https://www.inspiredtaste.net/wp-content/uploads/2025/07/Pancake-Recipe-1.jpg"
prompt = "Extract the subject and the background of this image and output that into a bullet list."

output_processed_image = process_image(prompt, image_of_a_pancake)
structured_output = extract_structured_text(output_processed_image)

Markdown(f"""
**Here's a structured summary of the image:**
- **Subject:** {structured_output.subject}
- **Background:** {structured_output.general_background_description}
""")

## Legacy: the Chat Completions API

Chat Completions still works with `gpt-5.5`, but it is now the **legacy** surface. The conversation-state, server-side compaction, response `phase`, connectors, hosted tools, and WebSocket features above are **Responses-API-first** and are not available through Chat Completions.

Use Chat Completions only when porting old code; new work should target the Responses API.

In [ ]:
# Legacy Chat Completions (shown for migration reference only)
completion = client.chat.completions.create(
    model="gpt-5.5",
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Hello!"},
    ],
)
print(completion.choices[0].message.content)